In [1]:
import pandas as pd

src = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_20260120.csv"
dst = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_LFP_20260204.csv"

df = pd.read_csv(src)
df_out = df[df["id"].isin(["AM", "AN", "AO"])]
df_out.to_csv(dst, index=False)

In [4]:
path_in = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_LFP_20260204.csv"
path_out = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/trial_info_LFP_20260205.csv"

df = pd.read_csv(path_in)

rows = []
n = len(df)
for i, row in df.iterrows():
    rows.append(row)
    skip = (row["trial_type"] == "H") and (str(row["trial_type_day"]) in ["1", "2"])
    if not skip:
        new_row = row.copy()
        new_row["trial_type"] = "B"
        rows.append(new_row)

    if i < n - 1:
        cur_day = str(row["trial_type_day"])
        next_row = df.loc[i + 1]
        next_day = str(next_row["trial_type_day"])
        if cur_day != next_day:
            skip_extra = (next_row["trial_type"] == "H") and (str(next_row["trial_type_day"]) in ["1", "2"])
            if not skip_extra:
                extra_row = next_row.copy()
                extra_row["trial_id"] = 0
                extra_row["trial_type"] = "B"
                rows.append(extra_row)

df_out = pd.DataFrame(rows).reset_index(drop=True)

# Add ephys_trial_id per session (starts at 1 for each ses)
df_out["ephys_trial_id"] = df_out.groupby("ses").cumcount() + 1

# Insert ephys_trial_id between usable and trial_note
cols = df_out.columns.tolist()
if "ephys_trial_id" in cols:
    cols.remove("ephys_trial_id")
if "trial_note" in cols:
    insert_idx = cols.index("trial_note")
    cols.insert(insert_idx, "ephys_trial_id")
    df_out = df_out[cols]

df_out.to_csv(path_out, index=False)